# RSNA Knee Abnormality Detection — Engine 9 Fold 1: ConvNeXt-Small 384px 2.5D MIL
### Platform: Google Colab Pro (A100 / L4 GPU)
**Key Innovations:**
- **Fold 1 Complementary Split:** Trains on the disjoint 80% training slice to create the official 2-Fold Sovereign Ensemble.
- **Native 384x384 2.5D MIL:** 3-slice contiguous triplets preserving 100% of ImageNet-22k pretrained features.
- **Pathology-Specific Attention MIL Head:** 12 independent attention heads weighting slice importance per abnormality.
- **Monitored on Gold-58:** Evaluated against official radiologist gold labels after every epoch.


In [ ]:
# 1. Environment & GPU Diagnostic
!nvidia-smi
!pip install -q timm pyarrow kaggle
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Authenticate Kaggle & Download Training Assets (~90 seconds)
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)

# Embedded credentials (Zero manual input required)
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_bf1b2382f2c6c8823e02df74217ba8e2'
os.environ['KAGGLE_USERNAME'] = 'cyruspinto'
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': 'cyruspinto', 'key': 'KGAT_bf1b2382f2c6c8823e02df74217ba8e2'}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle Authenticated as cyruspinto!')

os.makedirs('/content/corpus', exist_ok=True)
os.makedirs('/content/labels', exist_ok=True)
os.makedirs('/content/rsna_comp', exist_ok=True)

!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus -p /content/corpus --unzip -q
!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus-ext -p /content/corpus --unzip -q
!kaggle datasets download -d cyruspinto/rsna-knee-super-consensus-labels-0916 -p /content/labels --unzip -q
!kaggle competitions download -c rsna-knee-abnormality-detection -f train.csv -p /content/rsna_comp -q
!unzip -q -o /content/rsna_comp/train.csv.zip -d /content/rsna_comp/ 2>/dev/null || true
print('All training corpora downloaded and verified!')


In [ ]:
# 3. Write Engine 9 Fold 1 Training Script
with open('train_engine9_fold1.py', 'w', encoding='utf-8') as f:
    f.write('''"""
RSNA Knee Abnormality Detection — Engine 9 Fold 1: ConvNeXt-Small 384px 2.5D MIL
Autonomous GPU Training Pipeline with Attention MIL Head & Fold 1 Split
"""

import os, sys, time, math, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

def fast_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(bool)
    y_score = np.asarray(y_score)
    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = np.empty(len(y_score), dtype=float)
    order = np.argsort(y_score)
    ranks[order] = np.arange(len(y_score))
    _, inv, counts = np.unique(y_score, return_inverse=True, return_counts=True)
    if np.any(counts > 1):
        tie_ranks = np.zeros(len(counts))
        np.add.at(tie_ranks, inv, ranks)
        tie_ranks = tie_ranks / counts
        ranks = tie_ranks[inv]
    rank_sum = np.sum(ranks[y_true])
    u = rank_sum - (n_pos * (n_pos - 1)) / 2.0
    return float(u / (n_pos * n_neg))

class TwoPartVols:
    def __init__(self, a, b):
        self.a, self.b = a, b
        self.n_a = a.shape[0]
        self.shape = (a.shape[0] + (b.shape[0] if b is not None else 0),) + tuple(a.shape[1:])
    def __len__(self): return self.shape[0]
    def __getitem__(self, row):
        return self.a[row] if row < self.n_a else self.b[row - self.n_a]

class MIL384Dataset(Dataset):
    def __init__(self, ids, id2row, labels, vols, res=384, train=True, n_slices=8):
        self.ids = ids
        self.id2row = id2row
        self.labels = labels
        self.vols = vols
        self.res = res
        self.train = train
        self.n_slices = n_slices

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        uid = self.ids[i]
        row = self.id2row[uid]
        vol = self.vols[row]  # shape: (D, H, W)
        D = vol.shape[0]

        if D <= 4:
            centers = [min(D - 1, max(1, k)) for k in range(self.n_slices)]
        else:
            step = (D - 2) / float(self.n_slices + 1)
            if self.train:
                jitter = (random.random() - 0.5) * step * 0.5
                centers = [int(round(1 + (k + 1) * step + jitter)) for k in range(self.n_slices)]
            else:
                centers = [int(round(1 + (k + 1) * step)) for k in range(self.n_slices)]
            centers = [max(1, min(D - 2, c)) for c in centers]

        triplets = []
        for c in centers:
            trip = vol[c-1:c+2].astype(np.float32) / 255.0  # (3, H, W)
            if trip.shape[0] < 3:
                trip = np.repeat(vol[c:c+1].astype(np.float32) / 255.0, 3, axis=0)
            triplets.append(trip)

        stack = np.stack(triplets, axis=0)  # (n_slices, 3, H, W)
        t = torch.from_numpy(np.ascontiguousarray(stack))

        if t.shape[-1] != self.res or t.shape[-2] != self.res:
            t = F.interpolate(t, size=(self.res, self.res), mode='bilinear', align_corners=False)

        if self.train:
            if random.random() < 0.5:
                t = torch.flip(t, dims=[-1])
            gain = 1.0 + (random.random() - 0.5) * 0.12
            t = (t * gain).clamp(0, 1)

        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        t = (t - mean) / std

        y = torch.tensor(self.labels[uid], dtype=torch.float32)
        return t, y

class Engine9MIL(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('convnext_small.fb_in22k_ft_in1k_384', pretrained=True, num_classes=0)
        self.norm = nn.LayerNorm(768)
        self.att = nn.Sequential(
            nn.Linear(768, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 12)
        )
        self.clsW = nn.Parameter(torch.empty(12, 768))
        self.clsb = nn.Parameter(torch.empty(12))
        nn.init.xavier_uniform_(self.clsW)
        nn.init.zeros_(self.clsb)

    def forward(self, x):
        # x: (B, S, 3, 384, 384)
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        h = self.backbone(x)  # (B*S, 768)
        h = self.norm(h)
        h = h.view(B, S, 768)

        att_logits = self.att(h)  # (B, S, 12)
        alpha = F.softmax(att_logits, dim=1)  # (B, S, 12)

        # z: (B, 12, 768)
        z = torch.einsum('bsk,bsd->bkd', alpha, h)
        logits = (z * self.clsW.unsqueeze(0)).sum(dim=-1) + self.clsb.unsqueeze(0)  # (B, 12)
        return logits

def main():
    print('=== Starting RSNA Knee Engine 9 Fold 1: ConvNeXt-Small 384px 2.5D MIL ===', flush=True)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}', flush=True)

    # Load Corpus
    va = np.load('/content/corpus/all_vols.npy', mmap_mode='r')
    vb = np.load('/content/corpus/extra_vols.npy', mmap_mode='r') if os.path.exists('/content/corpus/extra_vols.npy') else None
    vols = TwoPartVols(va, vb)

    a_ids = np.load('/content/corpus/all_ids.npy', allow_pickle=True).astype(str)
    b_ids = np.load('/content/corpus/extra_ids.npy', allow_pickle=True).astype(str) if os.path.exists('/content/corpus/extra_ids.npy') else np.array([])
    all_ids = np.concatenate([a_ids, b_ids]) if len(b_ids) > 0 else a_ids

    id2row = {u: i for i, u in enumerate(all_ids)}
    idset = set(all_ids)
    print(f'Loaded corpus: {len(all_ids)} examinations.', flush=True)

    # Load Gold Anchor & Continuous Soft Labels
    tr = pd.read_csv('/content/rsna_comp/train.csv')
    tr['StudyInstanceUID'] = tr['StudyInstanceUID'].astype(str)
    gold_df = tr[tr[TARGETS].notna().all(axis=1)].copy().set_index('StudyInstanceUID')
    gold_ids = [u for u in gold_df.index if u in idset]
    print(f'Anchored on {len(gold_ids)} official radiologist gold studies.', flush=True)

    soft_path = '/content/labels/SUPER_CONSENSUS_labels.parquet'
    soft = pd.read_parquet(soft_path).set_index('StudyInstanceUID')

    train_ids = [u for u in all_ids if u in soft.index and u not in set(gold_ids)]
    labels = {u: soft.loc[u, TARGETS].values.astype(np.float32) for u in train_ids}
    for u in gold_ids:
        labels[u] = gold_df.loc[u, TARGETS].values.astype(np.float32)

    # 5-fold split: FOLD 1 (disjoint from Fold 0)
    np.random.seed(42)
    shuffled_train = list(train_ids)
    np.random.shuffle(shuffled_train)
    n_val = len(shuffled_train) // 5
    val_ids = shuffled_train[n_val:2*n_val]
    core_train_ids = shuffled_train[:n_val] + shuffled_train[2*n_val:]
    print(f'Fold 1 Split: {len(core_train_ids)} train, {len(val_ids)} validation.', flush=True)

    tds = MIL384Dataset(core_train_ids, id2row, labels, vols, res=384, train=True, n_slices=8)
    vds = MIL384Dataset(val_ids, id2row, labels, vols, res=384, train=False, n_slices=8)
    gds = MIL384Dataset(gold_ids, id2row, labels, vols, res=384, train=False, n_slices=8)

    # Batch size 12 on A100 SXM4 (12 * 8 slices = 96 forward passes per batch)
    tl = DataLoader(tds, batch_size=12, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    vl = DataLoader(vds, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
    gl = DataLoader(gds, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

    print('Initializing ConvNeXt-Small (in22k pretrained) for 384px MIL...', flush=True)
    model = Engine9MIL().to(device)

    epochs = 8
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(tl), eta_min=1e-6)
    lossf = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda')

    best_gold_auc = 0.0
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        tot = 0.0
        for x, y in tl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                logits = model(x)
                loss = lossf(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()

        # Gold Evaluation
        model.eval()
        P, Y = [], []
        with torch.no_grad():
            for gx, gy in gl:
                with torch.amp.autocast('cuda'):
                    gl_out = model(gx.to(device))
                P.append(torch.sigmoid(gl_out).float().cpu().numpy())
                Y.append(gy.numpy())
        P = np.concatenate(P, axis=0)
        Y = np.concatenate(Y, axis=0)
        gold_aucs = [fast_roc_auc(Y[:, j], P[:, j]) for j in range(len(TARGETS))]
        macro_gold = float(np.nanmean(gold_aucs))

        print(f'Epoch {ep+1:2d}/{epochs} | Loss: {tot/len(tl):.4f} | Gold-58 Macro AUC: {macro_gold:.4f} (Best: {max(best_gold_auc, macro_gold):.4f}) | {time.time()-t0:.0f}s', flush=True)
        for target_name, target_auc in zip(TARGETS, gold_aucs):
            print(f'   {target_name:18s}: {target_auc:.4f}', flush=True)

        if macro_gold > best_gold_auc:
            best_gold_auc = macro_gold
            torch.save({
                'epoch': ep + 1,
                'model': model.state_dict(),
                'gold_auc': macro_gold,
                'aucs': dict(zip(TARGETS, gold_aucs)),
                'arch': 'convnext_small_mil_384',
                'targets': TARGETS
            }, '/content/engine9_convnext_384_fold1_best.pt')
            print(f'  -> New Best Saved with Gold AUC: {macro_gold:.4f}', flush=True)

    print(f'\\nTraining Finished! Peak Gold-58 Macro-AUC: {best_gold_auc:.4f}', flush=True)

if __name__ == '__main__':
    main()
''')
print('Wrote train_engine9_fold1.py successfully!')


In [ ]:
# 4. Launch Autonomous GPU Training
!python train_engine9_fold1.py


In [ ]:
# 5. Package & Publish Fold 1 Weights to Kaggle Datasets
import os, json
os.makedirs('/content/kaggle_engine9_fold1', exist_ok=True)
!cp /content/engine9_convnext_384_fold1_best.pt /content/kaggle_engine9_fold1/

dataset_meta = {
    'title': 'RSNA Knee Engine 9 Fold 1 Weights',
    'id': 'cyruspinto/rsna-knee-engine9-fold1-weights',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open('/content/kaggle_engine9_fold1/dataset-metadata.json', 'w') as f:
    json.dump(dataset_meta, f, indent=2)

!kaggle datasets create -p /content/kaggle_engine9_fold1/ -u --dir-mode zip || kaggle datasets version -p /content/kaggle_engine9_fold1/ -m 'Update Engine 9 Fold 1 Weights' --dir-mode zip
print('Engine 9 Fold 1 Weights published to Kaggle successfully!')
